In [ ]:
"puissance 4"

import tkinter as tk
import random
import json
TAILLE_CASE = 90
NB_LIGNES = 6
NB_COLONNES = 7
historique = []
grille = [[0 for _ in range(NB_COLONNES)] for _ in range(NB_LIGNES)]
joueur_actuel = random.randint(1,2)
racine = tk.Tk()
racine.title("Puissance 4")

if joueur_actuel == 1:
    couleur_depart = "Rouge"
else:
    couleur_depart = "Jaune"


texte_info = tk.Label(racine, text=f"Le tirage au sort a désigné le joueur {couleur_depart} pour commencer !", font=("Arial", 14))
texte_info.pack()


canvas = tk.Canvas(racine, width=NB_COLONNES*TAILLE_CASE, height=NB_LIGNES*TAILLE_CASE, bg="blue")
canvas.pack()



def dessiner_plateau():
    for l in range(NB_LIGNES):
        for c in range(NB_COLONNES):
            x1, y1 = c * TAILLE_CASE + 10, l * TAILLE_CASE + 10
            x2, y2 = (c + 1) * TAILLE_CASE - 10, (l + 1) * TAILLE_CASE - 10
            canvas.create_oval(x1, y1, x2, y2, fill="white", outline="black")
dessiner_plateau()

            
def gerer_clic(event):
    global joueur_actuel
    colonne = event.x // TAILLE_CASE
    #print('tu as cliqué sur la colonne:', colonne)
    for ligne in range(NB_LIGNES-1, -1, -1):
        if grille[ligne][colonne] ==0:
            if joueur_actuel == 1:
                nom_joueur = "Rouge"
                couleur_pion = "red"
                prochain = "jaune"
            else:
                nom_joueur = "Jaune"
                couleur_pion = "yellow"
                prochain = "rouge"

            grille[ligne][colonne] = joueur_actuel
            historique.append((ligne, colonne))
            x1, y1 = colonne * TAILLE_CASE + 10, ligne * TAILLE_CASE + 10
            x2, y2 = (colonne + 1) * TAILLE_CASE - 10, (ligne + 1) * TAILLE_CASE - 10
            canvas.create_oval(x1, y1, x2, y2, fill=couleur_pion, outline="black")

            if verifier_victoire(joueur_actuel): 
                texte_info.config(text=f"Félicitations ! Le joueur {nom_joueur} a gagné !", fg=couleur_pion)
                canvas.unbind("<Button-1>")
                return 
            
            if grille_pleine():
                texte_info.config(text="Match nul ! La grille est pleine.", fg="black")
                canvas.unbind("<Button-1>")
                return

            if joueur_actuel == 1:
                joueur_actuel = 2 
            else:
                joueur_actuel = 1 
            
            texte_info.config(text=f"Au tour du joueur {prochain}")
            break


def verifier_victoire(joueur):
    for l in range (NB_LIGNES):
        for c in range(NB_COLONNES - 3):
            if grille[l][c] == grille[l][c+1] == grille[l][c+2] == grille[l][c+3] == joueur:
                return True
            
    for l in range (NB_LIGNES - 3):
        for c in range(NB_COLONNES):
            if grille[l][c] == grille[l+1][c] == grille[l+2][c] == grille[l+3][c] == joueur:
                return True
            
    for l in range (NB_LIGNES - 3):
        for c in range(NB_COLONNES - 3):
            if grille[l][c] == grille[l+1][c+1] == grille[l+2][c+2] == grille[l+3][c+3] == joueur:
                return True
            
    for l in range (3 , NB_LIGNES):
        for c in range(NB_COLONNES - 3):
            if grille[l][c] == grille[l-1][c+1] == grille[l-2][c+2] == grille[l-3][c+3] == joueur:
                return True
    return False

def grille_pleine():
    for l in range(NB_LIGNES):
        for c in range(NB_COLONNES):
            if grille[l][c] == 0:
                return False
    return True 

def annuler_coup():
    global joueur_actuel
    if len(historique) > 0:
        l, c = historique.pop()
    
        grille[l][c] = 0
        x1, y1 = c * TAILLE_CASE + 10, l * TAILLE_CASE + 10
        x2, y2 = (c + 1) * TAILLE_CASE - 10, (l + 1) * TAILLE_CASE - 10
        canvas.create_oval(x1, y1, x2, y2, fill="white", outline="black")
        
        if joueur_actuel == 1:
            joueur_actuel = 2
            nom = "jaune"
        else:
            joueur_actuel = 1
            nom = "rouge"
        
        texte_info.config(text=f"Coup annulé ! Au tour du joueur {nom}")
        
        canvas.bind("<Button-1>", gerer_clic)

def sauvegarder_partie():
    donnees = {
        "grille": grille,
        "joueur": joueur_actuel,
        "historique": historique
    }
    with open("sauvegarde_p4.json", "w") as f:
        json.dump(donnees, f)
    texte_info.config(text="Partie sauvegardée !", fg="blue")

def charger_partie():
    global grille, joueur_actuel, historique
    try:
        with open("sauvegarde_p4.json", "r") as f:
            donnees = json.load(f)
        
        grille = donnees["grille"]
        joueur_actuel = donnees["joueur"]
        historique = donnees["historique"]
        
        
        for l in range(NB_LIGNES):
            for c in range(NB_COLONNES):
                x1, y1 = c * TAILLE_CASE + 10, l * TAILLE_CASE + 10
                x2, y2 = (c + 1) * TAILLE_CASE - 10, (l + 1) * TAILLE_CASE - 10
                if grille[l][c] == 1:
                    canvas.create_oval(x1, y1, x2, y2, fill="red", outline="black")
                elif grille[l][c] == 2:
                    canvas.create_oval(x1, y1, x2, y2, fill="yellow", outline="black")
                else:
                    canvas.create_oval(x1, y1, x2, y2, fill="white", outline="black")
        
        nom = "Rouge" if joueur_actuel == 1 else "Jaune"
        texte_info.config(text=f"Partie chargée ! Au tour du joueur {nom}", fg="black")
        canvas.bind("<Button-1>", gerer_clic) 
        
    except FileNotFoundError:
        texte_info.config(text="Aucune sauvegarde trouvée...", fg="red")








canvas.bind("<Button-1>", gerer_clic)
bouton_undo = tk.Button(racine, text="Annuler le dernier coup", command=annuler_coup)
bouton_undo.pack()
cadre_boutons = tk.Frame(racine)
cadre_boutons.pack()

tk.Button(cadre_boutons, text="Sauvegarder", command=sauvegarder_partie).pack(side="left")
tk.Button(cadre_boutons, text="Charger", command=charger_partie).pack(side="left")
cadre_boutons = tk.Frame(racine)
cadre_boutons.pack()
racine.mainloop()





tu as cliqué sur la colonne: 0
tu as cliqué sur la colonne: 1
tu as cliqué sur la colonne: 2
tu as cliqué sur la colonne: 3
tu as cliqué sur la colonne: 4
tu as cliqué sur la colonne: 5
tu as cliqué sur la colonne: 6
tu as cliqué sur la colonne: 5
tu as cliqué sur la colonne: 4
tu as cliqué sur la colonne: 3
tu as cliqué sur la colonne: 1
tu as cliqué sur la colonne: 0
tu as cliqué sur la colonne: 1
tu as cliqué sur la colonne: 2
tu as cliqué sur la colonne: 3
tu as cliqué sur la colonne: 5
tu as cliqué sur la colonne: 5
tu as cliqué sur la colonne: 3
tu as cliqué sur la colonne: 4
tu as cliqué sur la colonne: 4
tu as cliqué sur la colonne: 4
tu as cliqué sur la colonne: 4
tu as cliqué sur la colonne: 3
tu as cliqué sur la colonne: 3
tu as cliqué sur la colonne: 0
tu as cliqué sur la colonne: 0
tu as cliqué sur la colonne: 0
tu as cliqué sur la colonne: 0
tu as cliqué sur la colonne: 1
tu as cliqué sur la colonne: 1
tu as cliqué sur la colonne: 1
tu as cliqué sur la colonne: 2
tu as cl